# Experiment 2: Fairly-Tuned Ablation Baselines + Cross-Seed Grad-CAM Audit

This notebook directly resolves the two limitations the manuscript's Discussion (Section 6) flags as most
consequential and still open:

1. **Unequal hyperparameter tuning in the ablation.** The original ablation baselines (Table 3) used a
   single-phase, 20-epoch budget while the fusion model got a two-phase, up-to-65-epoch protocol. This
   notebook gives each single-backbone baseline the **identical two-phase protocol** as the fusion model
   (same Phase 1/Phase 2 epoch budgets, same early stopping, same learning rates) — the "fairer ablation"
   the paper says is needed before treating the fusion gain as fully attributable to fusion itself.
2. **The Grad-CAM corner-attention finding was only checked on the seed-42 model.** This notebook re-runs
   the quantitative Grad-CAM audit (5 images/class, corner-vs-center scoring) for **all 5 seeds**, not just
   one, producing a properly aggregated (mean ± std across 25 images per class) answer to whether the
   VeryMildDemented corner-attention pattern is a stable property of the architecture or an artifact of one
   particular seed's model.

**What this notebook does NOT redo:** the fusion model's own headline results (Table 2) are not
re-computed here — those already came from a proper 5-seed run. This notebook trains the fusion model
again only because a freshly trained copy is needed per seed to run Grad-CAM on (Kaggle sessions don't
persist models between notebook runs), not because the fusion numbers themselves are in question.

**Kaggle setup:** same dataset as before (896/64/3200/2240 fingerprint), GPU T4x2 or P100.

**Estimated runtime:** ≈50–60 min/seed × 5 seeds ≈ 4.5–5 hours (fusion + two fairly-tuned baselines +
Grad-CAM audit, per seed). Comfortably inside your 25-hour quota. Save Version → Save & Run All (Commit).


In [ ]:
# 1. Environment
import os, json, time, random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import EfficientNetB0, ResNet50
import matplotlib.pyplot as plt
import matplotlib.cm as cmap_module
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))


In [ ]:
# 2. Locate dataset, build index (identical logic to the original notebook)
CLASSES = ["NonDemented", "VeryMildDemented", "MildDemented", "ModerateDemented"]
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
label_to_idx = {c: i for i, c in enumerate(CLASSES)}

def normalize(name):
    return name.lower().replace("_", "").replace("-", "").replace(" ", "")

canonical_by_norm = {normalize(c): c for c in CLASSES}
class_dirs = {}
for root, dirs, files in os.walk("/kaggle/input"):
    norm_root_name = normalize(os.path.basename(root))
    if norm_root_name in canonical_by_norm:
        if any(f.lower().endswith((".jpg", ".jpeg", ".png")) for f in files):
            canon = canonical_by_norm[norm_root_name]
            class_dirs.setdefault(canon, []).append(root)

assert all(c in class_dirs for c in CLASSES), "Could not find all 4 class folders — check dataset input."

records = []
for cls, dirs_for_cls in class_dirs.items():
    for cls_dir in dirs_for_cls:
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith((".jpg", ".jpeg", ".png")):
                records.append({"filepath": os.path.join(cls_dir, fname), "label": cls})
df = pd.DataFrame(records).drop_duplicates(subset="filepath").reset_index(drop=True)
print("Total images:", len(df))


In [ ]:
# 3. Identical fixed split (SPLIT_SEED=42) so results are directly comparable to the main experiment
SPLIT_SEED = 42
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SPLIT_SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SPLIT_SEED)
print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

from sklearn.utils.class_weight import compute_class_weight
class_labels = sorted(df["label"].unique())
cw = compute_class_weight(class_weight="balanced", classes=np.array(class_labels), y=train_df["label"])
class_weight_dict = {label_to_idx[k]: v for k, v in zip(class_labels, cw)}
print("Class weights:", class_weight_dict)


In [ ]:
# 4. Data pipeline (identical to the main notebook)
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label=0):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    return tf.cast(img, tf.float32)

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.1)
    img = tf.image.random_contrast(img, 0.9, 1.1)
    return img, label

def make_dataset(frame, training=False, seed=42):
    paths = frame["filepath"].values
    labels = np.array([label_to_idx[l] for l in frame["label"].values])
    labels_oh = tf.keras.utils.to_categorical(labels, num_classes=len(CLASSES))
    ds = tf.data.Dataset.from_tensor_slices((paths, labels_oh))
    ds = ds.map(lambda p, l: (load_image(p), l), num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(2048, seed=seed)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


In [ ]:
# 5. Model builders
def build_fusion_model():
    inputs = layers.Input(shape=(*IMG_SIZE, 3))
    eff_input = tf.keras.applications.efficientnet.preprocess_input(inputs)
    eff_base = EfficientNetB0(include_top=False, weights="imagenet", input_tensor=eff_input, pooling=None)
    eff_base._name = "efficientnetb0_branch"
    eff_features = layers.GlobalAveragePooling2D(name="eff_gap")(eff_base.output)

    res_input = tf.keras.applications.resnet50.preprocess_input(inputs)
    res_base = ResNet50(include_top=False, weights="imagenet", input_tensor=res_input, pooling=None)
    res_base._name = "resnet50_branch"
    res_features = layers.GlobalAveragePooling2D(name="res_gap")(res_base.output)

    for layer in eff_base.layers: layer.trainable = False
    for layer in res_base.layers: layer.trainable = False

    fused = layers.Concatenate(name="feature_fusion")([eff_features, res_features])
    x = layers.BatchNormalization()(fused)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(len(CLASSES), activation="softmax", name="predictions")(x)
    model = models.Model(inputs, outputs, name="EffResNet_FusionNet")
    return model, eff_base, res_base

def build_single_backbone_two_phase(base_ctor, preprocess_fn, name):
    """Single backbone with the SAME two-phase architecture depth as the fusion model's own branch
    treatment: GAP -> BN -> Dense(512) -> Dropout(0.4) -> Dense(128) -> Dropout(0.3) -> softmax,
    so the ablation isolates the effect of FUSION itself, not a confound of head depth."""
    inputs = layers.Input(shape=(*IMG_SIZE, 3))
    x_in = preprocess_fn(inputs)
    base = base_ctor(include_top=False, weights="imagenet", input_tensor=x_in, pooling=None)
    base._name = name + "_backbone"
    feat = layers.GlobalAveragePooling2D(name=name + "_gap")(base.output)
    x = layers.BatchNormalization()(feat)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(len(CLASSES), activation="softmax")(x)
    model = models.Model(inputs, outputs, name=name)
    for layer in base.layers: layer.trainable = False
    return model, base

def unfreeze_top(base, n_layers):
    for layer in base.layers[-n_layers:]:
        if not isinstance(layer, layers.BatchNormalization):
            layer.trainable = True

def evaluate_model(m, test_ds):
    y_true, y_pred = [], []
    for imgs, labels_oh in test_ds:
        probs = m.predict(imgs, verbose=0)
        y_pred.extend(np.argmax(probs, axis=1))
        y_true.extend(np.argmax(labels_oh.numpy(), axis=1))
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro")),
    }

def get_test_probs_and_labels(m, test_ds):
    # Returns (probs [n_test, n_classes], true_labels [n_test]) in the fixed, deterministic order
    # test_ds naturally iterates in (no shuffle applied to test_ds), so arrays align across seeds.
    all_probs, y_true = [], []
    for imgs, labels_oh in test_ds:
        probs = m.predict(imgs, verbose=0)
        all_probs.append(probs)
        y_true.extend(np.argmax(labels_oh.numpy(), axis=1))
    return np.concatenate(all_probs, axis=0), np.array(y_true)

def train_two_phase(model, base_list, train_ds, val_ds, phase1_epochs=25, phase2_epochs=40,
                     unfreeze_n_list=None):
    model.compile(optimizer=optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
    cb1 = [callbacks.EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True),
           callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6)]
    model.fit(train_ds, validation_data=val_ds, epochs=phase1_epochs,
              class_weight=class_weight_dict, callbacks=cb1, verbose=0)

    if unfreeze_n_list:
        for base, n in zip(base_list, unfreeze_n_list):
            unfreeze_top(base, n)
    model.compile(optimizer=optimizers.Adam(1e-5), loss="categorical_crossentropy", metrics=["accuracy"])
    cb2 = [callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True),
           callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-7)]
    model.fit(train_ds, validation_data=val_ds, epochs=phase2_epochs,
              class_weight=class_weight_dict, callbacks=cb2, verbose=0)
    return model


In [ ]:
# 6. Quantitative Grad-CAM audit function (same as the main notebook's fixed version)
def make_gradcam_heatmap(img_array, full_model, conv_layer_owner, last_conv_layer_name, pred_index=None):
    grad_model = models.Model(
        inputs=full_model.input,
        outputs=[conv_layer_owner.get_layer(last_conv_layer_name).output, full_model.output]
    )
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    grads = tape.gradient(class_channel, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_out[0]
    heatmap = conv_out @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index)

def corner_vs_center_score(heatmap_224):
    h, w = heatmap_224.shape
    cs = int(0.2 * h)
    corners = np.concatenate([
        heatmap_224[:cs, :cs].flatten(), heatmap_224[:cs, -cs:].flatten(),
        heatmap_224[-cs:, :cs].flatten(), heatmap_224[-cs:, -cs:].flatten(),
    ])
    c0, c1 = int(0.2 * h), int(0.8 * h)
    center = heatmap_224[c0:c1, c0:c1].flatten()
    return float(corners.mean()), float(center.mean())

def get_samples_for_class(model, cls_name, n=5, prefer_correct=True):
    rows = test_df[test_df["label"] == cls_name]
    cls_idx = label_to_idx[cls_name]
    chosen = []
    for _, row in rows.iterrows():
        if len(chosen) >= n:
            break
        arr = tf.expand_dims(load_image(row["filepath"]), 0)
        pred = model.predict(arr, verbose=0)
        if (not prefer_correct) or np.argmax(pred[0]) == cls_idx:
            chosen.append((row["filepath"], arr))
    if len(chosen) < n:
        for _, row in rows.iterrows():
            if len(chosen) >= n:
                break
            if row["filepath"] not in [c[0] for c in chosen]:
                chosen.append((row["filepath"], tf.expand_dims(load_image(row["filepath"]), 0)))
    return chosen

def run_gradcam_audit_for_seed(model, eff_base, seed):
    rows = []
    for cls_name in CLASSES:
        for fpath, arr in get_samples_for_class(model, cls_name, n=5):
            heatmap, pred_idx = make_gradcam_heatmap(arr, model, eff_base, "top_conv")
            heatmap_resized = tf.image.resize(heatmap[..., tf.newaxis], IMG_SIZE).numpy().squeeze()
            corner_score, center_score = corner_vs_center_score(heatmap_resized)
            rows.append({
                "seed": seed, "true_class": cls_name, "predicted_class": CLASSES[pred_idx],
                "corner_activation": corner_score, "center_activation": center_score,
                "corner_minus_center": corner_score - center_score,
            })
    return rows


In [ ]:
# 7. MAIN LOOP — for each seed: train fusion model (for Grad-CAM), run cross-seed Grad-CAM audit,
# then train BOTH single-backbone baselines with the SAME two-phase protocol (fair ablation).
SEEDS = [42, 123, 456, 789, 2024]

all_gradcam_rows = []
fair_ablation_results = []
all_seed_probs = []       # one [n_test, n_classes] probability array per seed, for the ensemble below
ensemble_y_true = None

gradcam_csv_path = "crossseed_gradcam_scores.csv"
ablation_csv_path = "fair_ablation_results.csv"
for p in [gradcam_csv_path, ablation_csv_path]:
    if os.path.exists(p):
        os.remove(p)

for i, seed in enumerate(SEEDS):
    print(f"\n===== Seed {seed} ({i+1}/{len(SEEDS)}) =====")
    t0 = time.time()
    tf.keras.backend.clear_session()
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

    train_ds = make_dataset(train_df, training=True, seed=seed)
    val_ds = make_dataset(val_df, training=False)
    test_ds = make_dataset(test_df, training=False)

    # --- Fusion model (needed fresh per seed to run Grad-CAM on) ---
    fusion_model, eff_base, res_base = build_fusion_model()
    fusion_model = train_two_phase(fusion_model, [eff_base, res_base], train_ds, val_ds,
                                    unfreeze_n_list=[30, 20])
    fusion_metrics = evaluate_model(fusion_model, test_ds)
    print(f"  Fusion (seed {seed}): accuracy={fusion_metrics['accuracy']:.4f} (sanity check vs main experiment)")

    # Save this seed's full test-set probability predictions for the 5-seed ensemble computed below.
    # Nearly free: this model is already being trained fresh for the Grad-CAM audit either way.
    seed_probs, seed_y_true = get_test_probs_and_labels(fusion_model, test_ds)
    all_seed_probs.append(seed_probs)
    if ensemble_y_true is None:
        ensemble_y_true = seed_y_true  # identical across seeds since test_ds order is deterministic
    np.save(f"test_probs_seed{seed}.npy", seed_probs)

    gradcam_rows = run_gradcam_audit_for_seed(fusion_model, eff_base, seed)
    all_gradcam_rows.extend(gradcam_rows)
    pd.DataFrame(gradcam_rows).to_csv(gradcam_csv_path, mode="a",
                                        header=not os.path.exists(gradcam_csv_path) or i == 0, index=False)

    del fusion_model, eff_base, res_base
    tf.keras.backend.clear_session()

    # --- Fairly-tuned EfficientNetB0-only baseline (same two-phase protocol as fusion) ---
    tf.random.set_seed(seed)
    eff_model, eff_base2 = build_single_backbone_two_phase(
        EfficientNetB0, tf.keras.applications.efficientnet.preprocess_input, "eff_only")
    eff_model = train_two_phase(eff_model, [eff_base2], train_ds, val_ds, unfreeze_n_list=[30])
    eff_metrics = evaluate_model(eff_model, test_ds)
    del eff_model, eff_base2
    tf.keras.backend.clear_session()

    # --- Fairly-tuned ResNet50-only baseline (same two-phase protocol as fusion) ---
    tf.random.set_seed(seed)
    res_model, res_base2 = build_single_backbone_two_phase(
        ResNet50, tf.keras.applications.resnet50.preprocess_input, "res_only")
    res_model = train_two_phase(res_model, [res_base2], train_ds, val_ds, unfreeze_n_list=[20])
    res_metrics = evaluate_model(res_model, test_ds)
    del res_model, res_base2
    tf.keras.backend.clear_session()

    elapsed = time.time() - t0
    result = {
        "seed": seed,
        "fusion_accuracy_sanity_check": fusion_metrics["accuracy"],
        "eff_only_fair_accuracy": eff_metrics["accuracy"], "eff_only_fair_f1_macro": eff_metrics["f1_macro"],
        "res_only_fair_accuracy": res_metrics["accuracy"], "res_only_fair_f1_macro": res_metrics["f1_macro"],
        "wall_clock_sec": round(elapsed, 1),
    }
    print(f"  EfficientNetB0-only (fair, seed {seed}): accuracy={eff_metrics['accuracy']:.4f}")
    print(f"  ResNet50-only (fair, seed {seed}): accuracy={res_metrics['accuracy']:.4f}")
    print(f"  Seed {seed} total time: {elapsed/60:.1f} min")
    fair_ablation_results.append(result)
    pd.DataFrame([result]).to_csv(ablation_csv_path, mode="a",
                                    header=not os.path.exists(ablation_csv_path) or i == 0, index=False)

print("\nAll seeds complete.")


In [ ]:
# 8. Aggregate: fair ablation mean ± std, for direct comparison against Table 3's original ablation
ablation_df = pd.DataFrame(fair_ablation_results)
print(ablation_df)

summary = {}
for col in ["fusion_accuracy_sanity_check", "eff_only_fair_accuracy", "eff_only_fair_f1_macro",
            "res_only_fair_accuracy", "res_only_fair_f1_macro"]:
    summary[col] = {"mean": float(ablation_df[col].mean()), "std": float(ablation_df[col].std())}

with open("fair_ablation_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nFair ablation mean ± std across 5 seeds:")
for col, v in summary.items():
    print(f"  {col}: {v['mean']:.4f} ± {v['std']:.4f}")

print("\nCompare eff_only_fair_accuracy / res_only_fair_accuracy above against the ORIGINAL")
print("(unfairly-tuned) ablation in the manuscript's Table 3: eff_only 55.42% ± 3.69%, res_only 57.08% ± 4.74%.")
print("If the fair versions are similar, the original gap was genuine. If the fair versions are much higher,")
print("the original ablation understated the baselines and Table 3 / the Discussion need revising accordingly.")


In [ ]:
# 9. Aggregate the cross-seed Grad-CAM audit — this is the key evidence for whether the VeryMildDemented
# corner-attention finding is a stable property of the architecture or specific to the seed-42 model.
gradcam_df = pd.DataFrame(all_gradcam_rows)
gradcam_df.to_csv("crossseed_gradcam_scores.csv", index=False)

agg = gradcam_df.groupby("true_class")[["corner_activation", "center_activation", "corner_minus_center"]].agg(["mean", "std"])
print("Cross-seed Grad-CAM audit (25 images per class: 5 seeds x 5 images):")
print(agg)

agg.to_csv("crossseed_gradcam_summary.csv")
print("\nCompare the VeryMildDemented corner_minus_center mean/std above against the seed-42-only result")
print("already in the manuscript (Table 6): +0.526 ± 0.270. If this cross-seed number is similarly positive")
print("with a comparable or tighter std, the finding is confirmed as robust across seeds, not a single-seed")
print("artifact, and Table 6 / Section 5.4 should be updated to report this stronger, multi-seed evidence.")


In [ ]:
# 10. 5-SEED ENSEMBLE: average the 5 independently trained fusion models' predicted probabilities
# (soft voting) and also compute majority-class hard voting, for comparison against the single-model
# mean accuracy already in the manuscript (90.56% ± 2.21%). This is a standard, well-established
# technique (deep ensembles) -- not a form of overfitting to the test set, since no test-set feedback
# was used to choose or tune any of the 5 individual models; they were trained independently beforehand
# and are combined here by simple averaging only.
from sklearn.metrics import precision_score, recall_score, roc_auc_score

stacked_probs = np.stack(all_seed_probs, axis=0)   # [5, n_test, n_classes]

# Soft voting: average probabilities across seeds, then argmax
soft_vote_probs = stacked_probs.mean(axis=0)        # [n_test, n_classes]
soft_vote_pred = np.argmax(soft_vote_probs, axis=1)

# Hard voting: each seed votes its own argmax class; majority wins (ties broken by summed probability)
per_seed_preds = np.argmax(stacked_probs, axis=2)   # [5, n_test]
hard_vote_pred = []
for i in range(per_seed_preds.shape[1]):
    votes = per_seed_preds[:, i]
    counts = np.bincount(votes, minlength=len(CLASSES))
    top = np.flatnonzero(counts == counts.max())
    if len(top) == 1:
        hard_vote_pred.append(top[0])
    else:
        # tie -> break using summed soft probability among tied classes
        hard_vote_pred.append(top[np.argmax(soft_vote_probs[i, top])])
hard_vote_pred = np.array(hard_vote_pred)

def summarize(y_true, y_pred, probs=None, name=""):
    y_true_oh = tf.keras.utils.to_categorical(y_true, num_classes=len(CLASSES))
    auc = None
    if probs is not None:
        try:
            auc = roc_auc_score(y_true_oh, probs, average="macro", multi_class="ovr")
        except Exception:
            pass
    result = {
        "method": name,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro")),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro")),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro")),
        "auc_macro": float(auc) if auc is not None else None,
    }
    print(name, "->", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in result.items()})
    return result

print("Single-model mean from the main 5-seed experiment (for reference): 90.56% ± 2.21% accuracy\n")
ensemble_results = []
ensemble_results.append(summarize(ensemble_y_true, soft_vote_pred, soft_vote_probs, "soft_vote_ensemble"))
ensemble_results.append(summarize(ensemble_y_true, hard_vote_pred, None, "hard_vote_ensemble"))

pd.DataFrame(ensemble_results).to_csv("ensemble_results.csv", index=False)
print("\nSaved ensemble_results.csv")
print("\nIf soft_vote_ensemble accuracy exceeds 90.56% (the single-model mean), report the ensemble as an")
print("additional, legitimate result in the manuscript -- e.g. 'a simple 5-seed averaging ensemble further")
print("improves accuracy to X%, consistent with standard deep-ensemble behavior' -- alongside, not instead")
print("of, the single-model mean ± std, since both numbers answer different questions (typical single-model")
print("performance vs. best achievable performance using all 5 trained models together).")


## What to send back

- `fair_ablation_results.csv` / `fair_ablation_summary.json` — the properly-tuned ablation baselines,
  for direct comparison against the manuscript's current Table 4.
- `crossseed_gradcam_scores.csv` / `crossseed_gradcam_summary.csv` — the Grad-CAM corner-vs-center audit
  across all 5 seeds (100 images total: 5 seeds × 4 classes × 5 images), for comparison against Table 6's
  current seed-42-only numbers.
- `ensemble_results.csv` — soft-vote and hard-vote 5-seed ensemble accuracy, for comparison against the
  90.56% ± 2.21% single-model mean already in the manuscript. If the ensemble beats the single-model mean
  (likely, based on standard deep-ensemble behavior), this becomes a genuine additional result worth adding
  to Section 5 — report it as what it is (an ensemble of the 5 already-trained models), not as a new
  single-model accuracy.

Send all three sets of files back and the manuscript's Table 4, Table 6, Section 5.1, and the Discussion
paragraphs that currently flag the ablation/Grad-CAM as open limitations can be updated with this stronger,
fully-resolved evidence.
